# EurecomGPT — Phase 2: train a tiny GPT & measure parallelism

Run this in **Google Colab** with a **GPU** runtime (Runtime -> Change runtime type -> T4 GPU). It exercises Lecture 3: CPU vectorization (SIMD), CPU threads, GPU (CUDA), and TPU. Fill the TODOs in `attention_numpy.py` and `model.py` first (see TASKS.md).

## 0. Setup — clone your repo and install deps

In [ ]:
# Edit the URL to YOUR repo.
!git clone https://github.com/<you>/<your-repo>.git repo 2>/dev/null || true
%cd repo
!pip -q install safetensors google-cloud-storage
import sys; sys.path.insert(0, 'phase-2-tiny-gpt')

In [ ]:
import numpy as np, torch
from attention_numpy import attention_naive, attention_vectorized
from config import CONFIG
from model import GPT
import train, experiments
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| torch', torch.__version__)

## 1-2. Attention: naive loops vs vectorized (SIMD)
The same math, timed two ways. `numpy.show_config()` confirms your NumPy uses an optimized BLAS with AVX/SIMD kernels.

In [ ]:
t_naive = experiments.time_attention(attention_naive, T=256, d=64)
t_vec   = experiments.time_attention(attention_vectorized, T=256, d=64)
print(f'naive      : {t_naive:.2f} ms')
print(f'vectorized : {t_vec:.2f} ms   ({t_naive / t_vec:.1f}x faster)')
np.show_config()

## 3. Multi-threading: sweep CPU threads (OpenMP-style intra-op parallelism)
PyTorch runs its CPU ops across `torch.set_num_threads(n)` threads. Sweep 1..8 and find where adding threads stops helping (the contention point).

In [ ]:
text = open('phase-2-tiny-gpt/input.txt', encoding='utf-8', errors='replace').read() \
       if __import__('os').path.exists('phase-2-tiny-gpt/input.txt') else \
       __import__('urllib.request', fromlist=['x']).urlopen(
           'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
       ).read().decode('utf-8', 'replace')
data = torch.tensor(train.encode(text), dtype=torch.long)
print('corpus tokens:', len(data))

In [ ]:
model = GPT(CONFIG)
print('parameters:', model.num_params())
def one_step():
    x, y = train.get_batch(data, CONFIG.block_size, 16, 'cpu')
    _, loss = model(x, y); loss.backward()
sweep = experiments.threading_sweep(one_step, threads=(1, 2, 4, 8))
print('threads -> ms:', sweep)

In [ ]:
import matplotlib.pyplot as plt
ts = sorted(int(k) for k in sweep)
plt.plot(ts, [sweep[str(t)] for t in ts], 'o-'); plt.xlabel('threads'); plt.ylabel('ms/step')
plt.title('CPU intra-op threading'); plt.grid(True); plt.show()

## 4. GPU (CUDA): the same training loop on a T4
Move the model + data to the GPU and watch the per-step time drop by ~50x or more.

In [ ]:
cpu_loss, cpu_s = train.train(GPT(CONFIG), data, steps=100, device='cpu', log_every=0)
gpu_model = GPT(CONFIG)
gpu_loss, gpu_s = train.train(gpu_model, data, steps=100, device=device, log_every=0)
print(f'CPU 100 steps: {cpu_s*1000:.0f} ms  |  {device} 100 steps: {gpu_s*1000:.0f} ms',
      f'  ({cpu_s/gpu_s:.1f}x)')

## 5. TPU (systolic arrays) — optional/best-effort
Switch the runtime to **TPU** and run with `torch_xla` (or JAX). If you have limited time, describe how a TPU's Matrix Unit (a systolic array, H.T. Kung) would execute the matmuls in attention, and why it wins on perf/W. Record any TPU timing you get.

In [ ]:
tpu_ms = None  # set to your measured ms/step if you ran on a TPU runtime
print('TPU ms/step:', tpu_ms)

## 6. Train the model, generate a sample, save weights

In [ ]:
model = GPT(CONFIG)
final_loss, secs = train.train(model, data, steps=2000, device=device, log_every=200)
print(f'final loss {final_loss:.3f} in {secs:.1f}s')
sample_text = train.sample(model, prompt='\n', max_new_tokens=200, device=device)
print('----- sample -----\n' + sample_text)
train.save_model(model, 'model.safetensors')

## 7. Upload to Cloud Storage and make it public
Store the weights in your GCS bucket and make the object public so the autograder can read it (a few MB, well under the 5 GB free tier).

In [ ]:
PROJECT = !gcloud config get-value project
PROJECT = PROJECT[0].strip()
BUCKET = f'{PROJECT}-eurecomgpt'   # bucket names are global; adjust if taken
!gcloud storage buckets create gs://{BUCKET} --location=US 2>/dev/null || true
!gcloud storage cp model.safetensors gs://{BUCKET}/model.safetensors
!gcloud storage objects update gs://{BUCKET}/model.safetensors --add-acl-grant=entity=allUsers,role=READER
GCS_URL = f'https://storage.googleapis.com/{BUCKET}/model.safetensors'
print(GCS_URL)

## 8. Write the submission report

In [ ]:
experiments.write_report(
    attention={'naive_ms': t_naive, 'vectorized_ms': t_vec, 'speedup': round(t_naive/t_vec, 2)},
    threads=sweep,
    devices={'cpu_ms': round(cpu_s*1000, 1), 'gpu_ms': round(gpu_s*1000, 1), 'tpu_ms': tpu_ms},
    training={'steps': 2000, 'final_loss': round(final_loss, 4), 'param_count': model.num_params()},
    sample_text=sample_text,
    gcs_url=GCS_URL,
)

Now **commit** `submission/phase2_report.json` and push (see TASKS.md). Your CI + the instructor grade the report + your public model.